# Olist E-Commerce Dataset — ETL Pipeline

In this notebook we clean and transform the raw Olist data 
to build the main analytical table for SQL analysis.

## Objectives
- Convert date columns to datetime
- Assign "sin_categoria" to products without category
- Translate product categories to English
- Filter only delivered orders
- Join all tables into a single analytical table
- Export clean data to SQL Server

In [2]:
import pandas as pd
import os

RAW_PATH = "../data/raw/"

tablas = {}
for archivo in os.listdir(RAW_PATH):
    nombre = archivo.replace(".csv", "").replace("olist_", "").replace("_dataset", "")
    tablas[nombre] = pd.read_csv(RAW_PATH + archivo)

print("Tablas cargadas:")
for nombre, df in tablas.items():
    print(f"  - {nombre}: {df.shape[0]:,} filas")

Tablas cargadas:
  - customers: 99,441 filas
  - geolocation: 1,000,163 filas
  - orders: 99,441 filas
  - order_items: 112,650 filas
  - order_payments: 103,886 filas
  - order_reviews: 99,224 filas
  - products: 32,951 filas
  - sellers: 3,095 filas
  - product_category_name_translation: 71 filas


In [3]:
# Convertir columnas de fecha a datetime
date_columns = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

orders = tablas['orders'].copy()

for col in date_columns:
    orders[col] = pd.to_datetime(orders[col])

print("Tipos de datos después de la conversión:")
print(orders.dtypes)

Tipos de datos después de la conversión:
order_id                                    str
customer_id                                 str
order_status                                str
order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object


In [4]:
# Filtrar solo ordenes delivered
orders_delivered = orders[orders['order_status'] == 'delivered'].copy()

print(f"Órdenes totales:     {len(orders):,}")
print(f"Órdenes delivered:   {len(orders_delivered):,}")
print(f"Órdenes descartadas: {len(orders) - len(orders_delivered):,}")

Órdenes totales:     99,441
Órdenes delivered:   96,478
Órdenes descartadas: 2,963


In [5]:
# Trabajar con productos
products = tablas['products'].copy()
translation = tablas['product_category_name_translation'].copy()

# Asignar "sin_categoria" a productos sin categoría
products['product_category_name'] = products['product_category_name'].fillna('sin_categoria')

# Join con la tabla de traducción
products = products.merge(translation, on='product_category_name', how='left')

# Los "sin_categoria" no tienen traducción, los completamos manualmente
products['product_category_name_english'] = products['product_category_name_english'].fillna('uncategorized')

print(f"Categorías únicas: {products['product_category_name_english'].nunique()}")
print(f"\nMuestra de categorías:")
print(products['product_category_name_english'].value_counts().head(10))

Categorías únicas: 72

Muestra de categorías:
product_category_name_english
bed_bath_table           3029
sports_leisure           2867
furniture_decor          2657
health_beauty            2444
housewares               2335
auto                     1900
computers_accessories    1639
toys                     1411
watches_gifts            1329
telephony                1134
Name: count, dtype: int64


In [6]:
print(products[products['product_category_name_english'] == 'uncategorized'].shape[0])

623


In [7]:
# Categorías sin traducción
sin_traduccion = products[
    (products['product_category_name_english'] == 'uncategorized') & 
    (products['product_category_name'] != 'sin_categoria')
]
print(sin_traduccion['product_category_name'].unique())

<StringArray>
['pc_gamer', 'portateis_cozinha_e_preparadores_de_alimentos']
Length: 2, dtype: str


In [8]:
# Traducir manualmente las categorías sin traducción
translation_manual = {
    'pc_gamer': 'pc_gamer',
    'portateis_cozinha_e_preparadores_de_alimentos': 'portable_kitchen_and_food_processors'
}

products['product_category_name_english'] = products.apply(
    lambda row: translation_manual.get(row['product_category_name'], row['product_category_name_english']),
    axis=1
)

# Verificar que no queden uncategorized con categoría en portugués
sin_traduccion = products[
    (products['product_category_name_english'] == 'uncategorized') & 
    (products['product_category_name'] != 'sin_categoria')
]
print(f"Categorías sin traducción restantes: {len(sin_traduccion)}")

Categorías sin traducción restantes: 0


In [9]:
# Construir tabla principal
df_main = orders_delivered.merge(tablas['order_items'], on='order_id', how='inner')
df_main = df_main.merge(products[['product_id', 'product_category_name_english']], on='product_id', how='left')
df_main = df_main.merge(tablas['customers'][['customer_id', 'customer_state', 'customer_unique_id']], on='customer_id', how='left')
df_main = df_main.merge(tablas['sellers'][['seller_id', 'seller_state']], on='seller_id', how='left')

print(f"Filas en tabla principal: {len(df_main):,}")
print(f"Columnas: {df_main.columns.tolist()}")

Filas en tabla principal: 110,197
Columnas: ['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value', 'product_category_name_english', 'customer_state', 'customer_unique_id', 'seller_state']


In [10]:
# Calcular columnas nuevas
df_main['delivery_time_days'] = (
    df_main['order_delivered_customer_date'] - df_main['order_purchase_timestamp']
).dt.days

df_main['estimated_delivery_days'] = (
    df_main['order_estimated_delivery_date'] - df_main['order_purchase_timestamp']
).dt.days

df_main['delay_days'] = df_main['delivery_time_days'] - df_main['estimated_delivery_days']
df_main['is_late'] = df_main['delay_days'] > 0
df_main['total_revenue'] = df_main['price'] + df_main['freight_value']

print(df_main[['delivery_time_days', 'estimated_delivery_days', 'delay_days', 'is_late', 'total_revenue']].describe())

       delivery_time_days  estimated_delivery_days     delay_days  \
count       110189.000000            110197.000000  110189.000000   
mean            12.007342                23.439631     -11.432285   
std              9.451153                 8.822024      10.168323   
min              0.000000                 2.000000    -146.000000   
25%              6.000000                18.000000     -17.000000   
50%             10.000000                23.000000     -12.000000   
75%             15.000000                28.000000      -7.000000   
max            209.000000               155.000000     189.000000   

       total_revenue  
count  110197.000000  
mean      139.929161  
std       189.319151  
min         6.080000  
25%        55.180000  
50%        92.130000  
75%       157.510000  
max      6929.310000  


In [11]:
# Agregar pagos - sumar todos los pagos por orden
payments = tablas['order_payments'].groupby('order_id').agg(
    total_payment=('payment_value', 'sum'),
    payment_installments=('payment_installments', 'max'),
    payment_type=('payment_type', 'first')
).reset_index()

df_main = df_main.merge(payments, on='order_id', how='left')

print(f"Filas después de agregar pagos: {len(df_main):,}")
print(f"\nTipos de pago:")
print(df_main['payment_type'].value_counts())

Filas después de agregar pagos: 110,197

Tipos de pago:
payment_type
credit_card    83351
boleto         22362
voucher         2829
debit_card      1652
Name: count, dtype: int64


In [12]:
# Agregar review score por orden
reviews = tablas['order_reviews'][['order_id', 'review_score']].drop_duplicates(subset='order_id')

df_main = df_main.merge(reviews, on='order_id', how='left')

print(f"Filas después de agregar reviews: {len(df_main):,}")
print(f"\nDistribución de review scores:")
print(df_main['review_score'].value_counts().sort_index())
print(f"\nReview score promedio: {df_main['review_score'].mean():.2f}")

Filas después de agregar reviews: 110,197

Distribución de review scores:
review_score
1.0    12475
2.0     3669
3.0     9183
4.0    21076
5.0    62967
Name: count, dtype: int64

Review score promedio: 4.08


In [13]:
# Eliminar columnas que no necesitamos
cols_to_drop = ['order_status', 'shipping_limit_date', 'customer_id']
df_main = df_main.drop(columns=cols_to_drop)

# Verificar estado final
print(f"Dimensiones finales: {df_main.shape}")
print(f"\nColumnas finales:")
for col in df_main.columns:
    print(f"  - {col}: {df_main[col].dtype}")

Dimensiones finales: (110197, 24)

Columnas finales:
  - order_id: str
  - order_purchase_timestamp: datetime64[us]
  - order_approved_at: datetime64[us]
  - order_delivered_carrier_date: datetime64[us]
  - order_delivered_customer_date: datetime64[us]
  - order_estimated_delivery_date: datetime64[us]
  - order_item_id: int64
  - product_id: str
  - seller_id: str
  - price: float64
  - freight_value: float64
  - product_category_name_english: str
  - customer_state: str
  - customer_unique_id: str
  - seller_state: str
  - delivery_time_days: float64
  - estimated_delivery_days: int64
  - delay_days: float64
  - is_late: bool
  - total_revenue: float64
  - total_payment: float64
  - payment_installments: float64
  - payment_type: str
  - review_score: float64


In [14]:
# Exportar tabla principal
EXPORT_PATH = "../exports/"
os.makedirs(EXPORT_PATH, exist_ok=True)

df_main.to_csv(EXPORT_PATH + "olist_main.csv", index=False)

print(f"Archivo exportado exitosamente.")
print(f"Tamaño: {os.path.getsize(EXPORT_PATH + 'olist_main.csv') / 1024 / 1024:.2f} MB")

Archivo exportado exitosamente.
Tamaño: 32.62 MB


In [15]:
from sqlalchemy import create_engine

# Conexión a SQL Server
engine = create_engine(
    "mssql+pyodbc://localhost/olist_ecommerce?driver=ODBC+Driver+17+for+SQL+Server&trusted_connection=yes"
)

# Cargar datos
df_main.to_sql('olist_main', con=engine, if_exists='append', index=False)

print(f"Datos cargados exitosamente: {len(df_main):,} filas")

c:\Users\Rami\AppData\Local\Programs\Python\Python313\Lib\site-packages\pandas\io\sql.py:1649: SAWarning: Unrecognized server version info '17.0.1000.7'.  Some SQL Server features may not function properly.
  con = self.exit_stack.enter_context(con.connect())


Datos cargados exitosamente: 110,197 filas
